In [1]:
import pandas as pd
from pyscenic.aucell import aucell
from pyscenic.utils import load_motifs
from pyscenic.utils import GeneSignature
import numpy as np
import os
import pickle
import scanpy as sc

/home/annikafomm/miniconda3/envs/aucell_env/lib/python3.10/site-packages/ctxcore/__init__.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [46]:
os.getcwd()

'/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend'

In [ ]:
data_dir = "../backend/data"

In [48]:
# Read matrix from mtx file
matrix = sc.read_mtx("/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend/data/CID3586/count_matrix_sparse.mtx").T
# Read genes and barcodes
genes = pd.read_csv("/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend/data/CID3586/count_matrix_genes.tsv", sep="\t", header=None)
barcodes = pd.read_csv("/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend/data/CID3586/count_matrix_barcodes.tsv", sep="\t", header=None)

In [49]:
anndata = sc.AnnData(X=matrix.X, obs=barcodes, var=genes)



/home/annikafomm/miniconda3/envs/aucell_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/annikafomm/miniconda3/envs/aucell_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [50]:
anndata.index = barcodes
anndata.var.columns = ["gene_id"]
anndata.obs.columns = ["cell_id"]
anndata.obs.index = anndata.obs.index.astype(str)
anndata.var.index = anndata.var.index.astype(str)
anndata.obs.columns = anndata.obs.columns.astype(str)
anndata.var.columns = anndata.var.columns.astype(str)

In [32]:
anndata = sc.read_h5ad("/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend/data/GSM6592049_M2_prepro_defaults.h5ad")

In [34]:
anndata.var.index = anndata.var['feature_name'].astype(str)

In [40]:
anndata.var.index

Index(['ENSG00000241860.7', 'ENSG00000286448.1', 'LINC01409', 'LINC01128',
       'LINC00115', 'FAM41C', 'LINC02593', 'SAMD11_ENSG00000187634',
       'NOC2L_ENSG00000188976', 'KLHL17_ENSG00000187961',
       ...
       'MT-CYB_ENSG00000198727', 'ENSG00000278704.1', 'ENSG00000277400.1',
       'MAFIP', 'ENSG00000276256.1', 'ENSG00000278384.1', 'ENSG00000273748.1',
       'ENSG00000271254.7', 'ENSG00000278817.1', 'ENSG00000277196.4'],
      dtype='object', name='feature_name', length=19210)

In [41]:

anndata.var.index = anndata.var.index.str.split("_").str[0]

In [42]:
anndata.var_names

Index(['ENSG00000241860.7', 'ENSG00000286448.1', 'LINC01409', 'LINC01128',
       'LINC00115', 'FAM41C', 'LINC02593', 'SAMD11', 'NOC2L', 'KLHL17',
       ...
       'MT-CYB', 'ENSG00000278704.1', 'ENSG00000277400.1', 'MAFIP',
       'ENSG00000276256.1', 'ENSG00000278384.1', 'ENSG00000273748.1',
       'ENSG00000271254.7', 'ENSG00000278817.1', 'ENSG00000277196.4'],
      dtype='object', name='feature_name', length=19210)

In [43]:
if 'feature_name' in anndata.var.columns:
    anndata.var = anndata.var.drop(columns=['feature_name'])

In [44]:
anndata.write_h5ad("/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend/data/GSM6592049_M2_prepro_defaults_gene_ids.h5ad")

In [33]:
# Filter
sc.pp.filter_cells(anndata, min_genes=200)
sc.pp.filter_genes(anndata, min_cells=3)

In [ ]:
# convert to df
df = pd.DataFrame(anndata.X.toarray(), index=anndata.obs.index, columns=anndata.var['gene_id'])

In [57]:
# Read genie3 regulons
genie3 = pd.read_csv(
   f"{data_dir}/genie3_BRCA_mrn.top_100k.csv",
    index_col=0,
)

In [58]:
# Create GeneSets from genie3 regulons
genesets = []
for regulator, group in genie3.groupby("regulatoryGene"):
    gene2weight = dict(zip(group["targetGene"], group["weight"]))
    geneset = GeneSignature(name=regulator, gene2weight=gene2weight)
    genesets.append(geneset)

In [59]:
# Overlap regulons with genes in the data
genie3_genes = set(genie3["targetGene"].unique())
genes_in_data = set(df.columns)
overlap_genes = genie3_genes.intersection(genes_in_data)
print(f"Number of genes in genie3 regulons: {len(genie3_genes)}")
print(f"Number of genes in data: {len(genes_in_data)}")
print(f"Number of overlapping genes: {len(overlap_genes)}")


Number of genes in genie3 regulons: 17339
Number of genes in data: 17944
Number of overlapping genes: 14159


In [60]:
genesets

[GeneSignature(name='ADNP', gene2weight=frozendict.frozendict({'TOP2B': 0.0279155260750008, 'EGFL7': 0.0237409127547856, 'RASIP1': 0.0206077146851765, 'ABCD4': 0.0198264699957939, 'CLDN5': 0.0196923416113006, 'DCLRE1A': 0.0196791886852008, 'UBXN2B': 0.0186096779888699, 'DHX40': 0.0175653551076977, 'FEM1B': 0.0169275218517057, 'UPP1': 0.0167412548801931, 'INHBC': 0.0164379980511868, 'ZNF146': 0.0162366041101673, 'CLDN15': 0.0162188588231912, 'SUV420H1': 0.0162154303199169, 'FBXO21': 0.0161746744825105, 'LYL1': 0.0161460409470039, 'GTF3C4': 0.0159948001546787, 'CHML': 0.0155638331522917, 'PPP2R5B': 0.0153062509862445, 'EXOC3L2': 0.015090823235948, 'FES': 0.0149996808025288, 'RBAK': 0.014902061047451, 'KAT7': 0.0148831899298347, 'RASSF1': 0.014873103380482, 'TADA1': 0.0147882868342616, 'RBBP4': 0.0146250663555892, 'PLD3': 0.0145239365173512, 'TMEM150A': 0.014377200370458, 'IKBKG': 0.0143513116808776, 'SIN3A': 0.013912341210591, 'TNFRSF1A': 0.0138251453401087, 'SLC27A3': 0.0137148123313428

In [68]:
aucell_results = aucell(
    df,
    genesets,
    num_workers=8
)

Less than 80% of the genes in ZNF366 are present in the expression matrix.
Less than 80% of the genes in ETV5 are present in the expression matrix.
Less than 80% of the genes in AKAP8 are present in the expression matrix.
Less than 80% of the genes in FAM200B are present in the expression matrix.
Less than 80% of the genes in ALX4 are present in the expression matrix.
Less than 80% of the genes in ANKZF1 are present in the expression matrix.
Less than 80% of the genes in ARHGAP35 are present in the expression matrix.
Less than 80% of the genes in ARID3A are present in the expression matrix.
Less than 80% of the genes in JRK are present in the expression matrix.
Less than 80% of the genes in ARX are present in the expression matrix.
Less than 80% of the genes in ZFP92 are present in the expression matrix.
Less than 80% of the genes in ASH1L are present in the expression matrix.
Less than 80% of the genes in FOXA3 are present in the expression matrix.
Less than 80% of the genes in ONECUT

In [62]:
aucell_results.T.to_dict()

{'CID3586_AAGACCTCAGCATGAG': {'ADNP': 0.06631082742540305,
  'ADNP2': 0.0,
  'AEBP1': 0.032827553689601524,
  'AEBP2': 0.01029935082818859,
  'AHCTF1': 0.03957219177316128,
  'AHDC1': 0.03144760998814675,
  'AHR': 0.01814639680994034,
  'AKAP8': 0.0,
  'AKAP8L': 0.016926678795367224,
  'AKNA': 0.0,
  'ALX4': 0.0,
  'ANKZF1': 0.0,
  'AR': 0.0,
  'ARHGAP35': 0.0,
  'ARID2': 0.04116562896933704,
  'ARID3A': 0.0,
  'ARID3B': 0.05262515410443675,
  'ARID5A': 0.0,
  'ARID5B': 0.06514034220883745,
  'ARNT': 0.009575312958443076,
  'ARNT2': 0.0,
  'ARNTL': 0.0,
  'ARNTL2': 0.0,
  'ARX': 0.0,
  'ASCL2': 0.0,
  'ASH1L': 0.0,
  'ATF1': 0.02577589865068282,
  'ATF2': 0.0009372369590697373,
  'ATF3': 0.1291839094868963,
  'ATF4': 0.0,
  'ATF5': 0.0,
  'ATF6': 0.0,
  'ATF6B': 0.010951005663772051,
  'ATF7': 0.045144054154651676,
  'ATMIN': 0.0,
  'ATOH8': 0.05644457738856323,
  'BACH1': 0.0,
  'BACH2': 0.0,
  'BARX2': 0.0018916561496321656,
  'BATF': 0.013353098284081794,
  'BATF2': 0.07368509564743

In [63]:
# How many zero
zero_aucell = np.sum(aucell_results > 0.4)
print(f"Number of zero AUC values: {zero_aucell}")

Number of zero AUC values: Regulon
ADNP      0
ADNP2     0
AEBP1     0
AEBP2     0
AHCTF1    0
         ..
ZUFSP     0
ZXDA      0
ZXDB      0
ZXDC      0
ZZZ3      0
Length: 1317, dtype: int64


/home/annikafomm/miniconda3/envs/aucell_env/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:84: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [64]:
aucell_results

Regulon,ADNP,ADNP2,AEBP1,AEBP2,AHCTF1,AHDC1,AHR,AKAP8,AKAP8L,AKNA,...,ZSCAN30,ZSCAN31,ZSCAN32,ZSCAN5A,ZSCAN9,ZUFSP,ZXDA,ZXDB,ZXDC,ZZZ3
Cell,,,,,,,,,,,,,,,,,,,,,
CID3586_AAGACCTCAGCATGAG,0.066311,0.000000,0.032828,0.010299,0.039572,0.031448,0.018146,0.0,0.016927,0.000000,...,0.0,0.165391,0.0,0.141389,0.0,0.000000,0.049302,0.0,0.061885,0.014633
CID3586_AAGGTTCGTAGTACCT,0.055614,0.103654,0.023453,0.036327,0.042993,0.004243,0.039899,0.0,0.000000,0.006354,...,0.0,0.000000,0.0,0.105879,0.0,0.108778,0.031762,0.0,0.029070,0.049385
CID3586_ACCAGTAGTTGTGGCC,0.098148,0.000000,0.040855,0.008408,0.024558,0.012568,0.016474,0.0,0.009254,0.000000,...,0.0,0.000000,0.0,0.000000,0.0,0.214813,0.067953,0.0,0.048385,0.020069
CID3586_ACCCACTAGATGTCGG,0.070620,0.000000,0.029995,0.016507,0.042844,0.006145,0.037655,0.0,0.031917,0.000000,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.058402,0.0,0.073809,0.044475
CID3586_ACTGATGGTCAACTGT,0.072209,0.000000,0.039254,0.019546,0.026125,0.010591,0.014668,0.0,0.018282,0.059405,...,0.0,0.000000,0.0,0.103497,0.0,0.000000,0.000000,0.0,0.028676,0.044310
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CID3586_TTCCCAGCATCGGGTC,0.019461,0.005085,0.000000,0.031122,0.028259,0.027042,0.018802,0.0,0.017588,0.021438,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.013899,0.0,0.043457,0.017137
CID3586_TTCTACATCATGTAGC,0.003263,0.000000,0.000000,0.013695,0.031590,0.020214,0.011822,0.0,0.000000,0.015442,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.063116,0.0,0.065909,0.026355
CID3586_TTCTTAGAGAGTCGGT,0.000000,0.000000,0.005860,0.053089,0.024522,0.023085,0.041673,0.0,0.020006,0.000000,...,0.0,0.119205,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.071444,0.019471


In [65]:
{str(gs.name): tuple(gs.gene2weight.keys()) for gs in genesets}

{'ADNP': ('TOP2B',
  'EGFL7',
  'RASIP1',
  'ABCD4',
  'CLDN5',
  'DCLRE1A',
  'UBXN2B',
  'DHX40',
  'FEM1B',
  'UPP1',
  'INHBC',
  'ZNF146',
  'CLDN15',
  'SUV420H1',
  'FBXO21',
  'LYL1',
  'GTF3C4',
  'CHML',
  'PPP2R5B',
  'EXOC3L2',
  'FES',
  'RBAK',
  'KAT7',
  'RASSF1',
  'TADA1',
  'RBBP4',
  'PLD3',
  'TMEM150A',
  'IKBKG',
  'SIN3A',
  'TNFRSF1A',
  'SLC27A3',
  'PRIM1',
  'SLC8B1',
  'E2F4',
  'TOMM20',
  'PSTPIP1',
  'TSPAN4',
  'RP11-218L14.4',
  'HPCAL1',
  'TCIRG1',
  'PLCB1',
  'POLR1B',
  'NOS3'),
 'ADNP2': ('USP6NL', 'SLC29A3', 'KIAA0930', 'TTC17', 'SLC16A12', 'CD52'),
 'AEBP1': ('P4HA3',
  'TMEM119',
  'COL6A2',
  'ADGRA2',
  'GJC2',
  'CPXM2',
  'C3',
  'MMP2',
  'C1R',
  'PCDHGC3',
  'MMP23B',
  'FBLN1',
  'RCN3',
  'CHRD',
  'PCOLCE',
  'OLFML3',
  'CPZ',
  'ADCY3',
  'GRAP',
  'FLNC',
  'SFRP2',
  'MRC2',
  'MRPS35',
  'CALML4',
  'C1QTNF5',
  'ISLR',
  'OAF',
  'EMILIN2',
  'THBS3',
  'RAB3IL1',
  'GLIS2',
  'MMP23A',
  'ADAM33',
  'GLI1',
  'FBLN7',
  'GRID1

In [ ]:
# Save results in anndata
anndata.obsm["aucell"] = aucell_results
# Save genesets to anndata
genesets_dict = {gs.name: list(gs.gene2weight.keys()) for gs in genesets}



In [ ]:
anndata.uns["aucell_genesets"] = genesets_dict
# Save anndata
anndata.write(os.path.join(f"{data_dir}/aucell", "CID3586_aucell.h5ad"))